### Step 1: Install necesscary packages

In [23]:
# !pip install matplotlib
# !pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [24]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [25]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [26]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1: Generate 1,000 deterministic preference pairs

Generate 200 unique symbolic questions per category: addition, subtraction,
multiplication, exact division, and algebra. Arithmetic answers and algebra
solutions are computed directly. Each negative uses the handout's fixed refusal,
`Sorry, I do not know.`; no model inference or natural-language prompt variations
are used. A fixed random seed makes generation reproducible.

Each JSON object contains `negative` and `positive`, both beginning with the same
question. Positives follow the handout's answer-and-explanation format. Candidates
that would exceed 64 characters including the four trailing training newlines
are skipped, so the training helper will not truncate them. This length filtering
favours shorter examples. Division is exact, and algebra uses six equation forms.

Run Steps 2-4, then these cells and Step 5 in order. Task 1 itself only needs the
tokenizer from Step 2. Change `PAIR_COUNT` to `100_000` for the full dataset;
rerunning replaces the JSON. The current 1,000-pair pilot is for checking the
pipeline before scaling. The timing now measures deterministic generation and
JSON saving, not GPU inference.

Implementation assistance: OpenAI Codex.

In [27]:
from pathlib import Path
from collections import Counter

# Explicitly reset this if the kernel still holds the previous value of 128.
max_length = 64
PAIR_COUNT = 1_000  # Change to 100_000 for the full dataset.
PAIR_SEED = 42
PAIR_PATH = Path("pos_neg_pairs.json")  # Run from dpo/.
TIMING_PATH = Path("task1_timing.json")
PAIR_CATEGORIES = ("addition", "subtraction", "multiplication", "division", "algebra")
NEGATIVE_RESPONSE = "Sorry, I do not know."
RESPONSE_LIMIT = max_length - len("\n\n\n\n")
assert PAIR_COUNT > 0 and PAIR_COUNT % len(PAIR_CATEGORIES) == 0
print(f"Training sequence length: {max_length}; response budget: {RESPONSE_LIMIT}")

Training sequence length: 64; response budget: 60


In [28]:
def make_question(category, rng):
    # Broad enough ranges for 20k unique questions per category in the full run.
    a, b = rng.randint(0, 199), rng.randint(0, 199)
    if category == "addition":
        return f"{a}+{b}=?", a + b, f"{a}+{b}={a + b}"
    if category == "subtraction":
        return f"{a}-{b}=?", a - b, f"{a}-{b}={a - b}"
    if category == "multiplication":
        return f"{a}*{b}=?", a * b, f"{a}*{b}={a * b}"
    if category == "division":
        divisor = rng.randint(1, 199)
        dividend = divisor * b
        return f"{dividend}/{divisor}=?", b, f"{dividend}/{divisor}={b}"
    x, a = rng.randint(0, 199), rng.randint(1, 199)
    form = rng.randrange(6)
    if form == 0:
        return f"x+{a}={x+a},x=?", x, f"{x+a}-{a}={x}"
    if form == 1:
        return f"x-{a}={x-a},x=?", x, f"{x-a}+{a}={x}"
    if form == 2:
        return f"{a}-x={a-x},x=?", x, f"{a}-{a-x}={x}"
    if form == 3:
        return f"x*{a}={x*a},x=?", x, f"{x*a}/{a}={x}"
    if form == 4:
        return f"{a}*x={a*x},x=?", x, f"{a*x}/{a}={x}"
    # Choose the quotient first, so x/a=b has an integer solution.
    return f"x/{a}={x},x=?", x*a, f"{x}*{a}={x*a}"




def build_questions():
    rng = random.Random(PAIR_SEED)
    questions, seen = [], set()
    per_category = PAIR_COUNT // len(PAIR_CATEGORIES)
    for category in PAIR_CATEGORIES:
        accepted = 0
        for _ in range(PAIR_COUNT * 100):
            prompt, answer, calculation = make_question(category, rng)
            expression, result = calculation.split("=")
            explanation = f"{expression} equals {result}"
            positive = f"{prompt} The answer is {answer} because {explanation}."
            negative = f"{prompt} {NEGATIVE_RESPONSE}"
            if prompt in seen or max(len(positive), len(negative)) > RESPONSE_LIMIT:
                continue
            assert set(positive + negative) <= set(stoi), "Unsupported tokenizer character."
            questions.append(dict(category=category, prompt=prompt, answer=answer,
                                  calculation=calculation, positive=positive,
                                  negative=negative))
            seen.add(prompt)
            accepted += 1
            if accepted == per_category:
                break
        else:
            raise RuntimeError(f"Cannot fill {category}; review operand ranges and length limit.")
    rng.shuffle(questions)
    return questions


def save_pairs(pairs):
    temporary = PAIR_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(pairs, indent=2), encoding="utf-8")
    temporary.replace(PAIR_PATH)

In [29]:
started = time.perf_counter()
questions = build_questions()
pairs = [{"negative": q["negative"], "positive": q["positive"]} for q in questions]
assert len(pairs) == PAIR_COUNT
assert len({q["prompt"] for q in questions}) == PAIR_COUNT
assert all(len(text) + 4 <= max_length for pair in pairs for text in pair.values())
assert all(set(text) <= set(stoi) for pair in pairs for text in pair.values())
coverage = dict(Counter(q["category"] for q in questions))
assert all(count == PAIR_COUNT // len(PAIR_CATEGORIES) for count in coverage.values())
save_pairs(pairs)
elapsed_seconds = time.perf_counter() - started
timing = {
    "method": "deterministic arithmetic with fixed refusal negatives",
    "pairs": len(pairs), "seed": PAIR_SEED, "category_counts": coverage,
    "max_length": max_length, "model_inference_calls": 0,
    "elapsed_seconds": elapsed_seconds,
}
TIMING_PATH.write_text(json.dumps(timing, indent=2), encoding="utf-8")
print("Question coverage:", coverage)
print(f"Saved {len(pairs):,} pairs to {PAIR_PATH.resolve()}")
print("Negative response:", NEGATIVE_RESPONSE)
print(f"Generation and JSON saving: {elapsed_seconds:.3f} seconds; no model inference.")
print(json.dumps(pairs[:5], indent=2))

Question coverage: {'algebra': 200, 'division': 200, 'multiplication': 200, 'subtraction': 200, 'addition': 200}
Saved 1,000 pairs to C:\Users\yuhen\Documents\GithubRepos\NanoGPT-Math\dpo\pos_neg_pairs.json
Negative response: Sorry, I do not know.
Generation and JSON saving: 0.058 seconds; no model inference.
[
  {
    "negative": "x-105=-32,x=? Sorry, I do not know.",
    "positive": "x-105=-32,x=? The answer is 73 because -32+105 equals 73."
  },
  {
    "negative": "2054/13=? Sorry, I do not know.",
    "positive": "2054/13=? The answer is 158 because 2054/13 equals 158."
  },
  {
    "negative": "240/120=? Sorry, I do not know.",
    "positive": "240/120=? The answer is 2 because 240/120 equals 2."
  },
  {
    "negative": "153*30=? Sorry, I do not know.",
    "positive": "153*30=? The answer is 4590 because 153*30 equals 4590."
  },
  {
    "negative": "83-54=? Sorry, I do not know.",
    "positive": "83-54=? The answer is 29 because 83-54 equals 29."
  }
]


### Step 5: Load Data (**students are required to complete this part!**)

In [8]:
# Load the pairs generated in Task 1 (working directory: dpo/).
with open("pos_neg_pairs.json", "r", encoding="utf-8") as f:
    lines = json.load(f)
print(f"Loaded {len(lines):,} preference pairs.")

Loaded 1,000 preference pairs.


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# recommend to use the AdamW optimizer 

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples: 
        # ...
        # neg_logprob
        # pos_logprob 
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        # ...
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################